# Area reference analysis

Parse the Synopsys hierarchical area report and compute TCU area references. For `tcu_fp`, BF16 multiplier area is excluded by subtracting only the top-level `g_prod_<n>__bf16_mul` instance rows to avoid double-counting child rows.

In [1]:
from pathlib import Path
import re

import pandas as pd

REPORT = Path("tcu_area.rpt")

In [2]:
AREA_ROW_RE = re.compile(
    r"^(?P<cell>\S+)\s+"
    r"(?P<absolute_area>[0-9]+\.[0-9]+)\s+"
    r"(?P<percent_total>[0-9]+\.[0-9]+)\s+"
    r"(?P<local_combi>[0-9]+\.[0-9]+)\s+"
    r"(?P<local_noncombi>[0-9]+\.[0-9]+)\s+"
    r"(?P<local_blackboxes>[0-9]+\.[0-9]+)\s+"
    r"(?P<design>\S+)"
)


def load_area_report(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text().splitlines():
        match = AREA_ROW_RE.match(line)
        if not match:
            continue
        row = match.groupdict()
        for key in ["absolute_area", "percent_total", "local_combi", "local_noncombi", "local_blackboxes"]:
            row[key] = float(row[key])
        rows.append(row)
    return pd.DataFrame(rows)


area_df = load_area_report(REPORT)
area_df.head()

,cell,absolute_area,percent_total,local_combi,local_noncombi,local_blackboxes,design
0,VX_tcu_unit,1.042243e+06,100.0,56.745,0.0000,0.0,VX_tcu_unit
1,dispatch_unit,1.810622e+04,1.7,54.756,0.0000,0.0,VX_dispatch_unit_h_511_242_619
2,dispatch_unit/g_blocks_0__buf_out,1.805146e+04,1.7,0.000,0.0000,0.0,VX_elastic_buffer
3,dispatch_unit/g_blocks_0__buf_out/g_eb2_out_buf,1.535391e+03,0.1,1535.391,0.0000,0.0,VX_pipe_buffer
4,dispatch_unit/g_blocks_0__buf_out/g_eb2_stream...,1.651607e+04,1.6,3227.913,13263.5877,0.0,VX_stream_buffer_6324_1


In [3]:
fpint_pnr = 1261432
fpint_cell = 693436.0606

tcu_cell = 1042243.1261
tcu_fp_raw = 706797.1110
tcu_int = 291289.8649
tcu_dispatch = 18106.2176
tcu_gather = 12036.0237
tcu_switch = 13957.1638

In [4]:
tcu_fp_row = area_df.loc[area_df["cell"].eq("g_blocks_0__tcu_fp")].squeeze()

# Use only the BF16 multiplier instance roots. Summing every row containing
# "bf16_mul" would double-count the hierarchy under each multiplier.
bf16_mul_top = area_df[area_df["cell"].str.match(r"^g_blocks_0__tcu_fp/.*/g_prod_\d+__bf16_mul$")]
fp16_mul_top = area_df[area_df["cell"].str.match(r"^g_blocks_0__tcu_fp/.*/g_prod_\d+__fp16_mul$")]

assert abs(float(tcu_fp_row["absolute_area"]) - tcu_fp_raw) < 1e-3
assert len(bf16_mul_top) == 256
assert len(fp16_mul_top) == 256

tcu_fp_bf16_mul_area = bf16_mul_top["absolute_area"].sum()
tcu_fp_fp16_mul_area = fp16_mul_top["absolute_area"].sum()
tcu_fp_without_bf16_mul = tcu_fp_raw - tcu_fp_bf16_mul_area

tcu_area_summary = pd.DataFrame([
    {"name": "tcu_fp_raw", "area": tcu_fp_raw},
    {"name": "bf16_mul_top_area_excluded", "area": tcu_fp_bf16_mul_area},
    {"name": "tcu_fp_without_bf16_mul", "area": tcu_fp_without_bf16_mul},
    {"name": "fp16_mul_top_area_reference", "area": tcu_fp_fp16_mul_area},
])
tcu_area_summary

,name,area
0,tcu_fp_raw,706797.111
1,bf16_mul_top_area_excluded,168291.513
2,tcu_fp_without_bf16_mul,538505.598
3,fp16_mul_top_area_reference,170007.201


In [5]:
tcu_fp_without_bf16_mul

np.float64(538505.598)

In [6]:
693436.0606/538505.598

1.287704460595041